# Test retrieval LinearRAG hybrid v2

Chạy một cell dưới đây là đủ:

1. Convert `legal_graph_hybrid_v2` sang format retriever cũ bằng converter v2.
2. Build index BM25 + graph.
3. Test các query mẫu.

In [1]:
from pathlib import Path
import subprocess, sys, json

# Nếu notebook đặt ở root KLTN/traffic-bot thì để ".".
# Nếu notebook đặt trong một thư mục con thì sửa ROOT lại cho đúng.
ROOT = Path(".").resolve()

DATA = ROOT / "data"
PRE = DATA / "preprocessed"
GRAPHS = DATA / "graphs"

CONVERTER = ROOT / "graph_hybrid_to_linearrag_compat_v2" / "convert_graph_to_linearrag_compat_v2.py"
RETRIEVER_DIR = ROOT / "legal_linearrag_retriever"

GRAPH_ROOT = GRAPHS / "legal_graph_hybrid_v2"
COMPAT_GRAPH = PRE / "legal_graph_hybrid_v2_linearrag_compat_v2"
GAZETTEER_ROOT = PRE / "gazetteers_v2"
INDEX_DIR = PRE / "linearrag_index_hybrid_v2_bm25_graph_v2"

QUERIES = [
    "Thời gian lái xe liên tục tối đa của người lái xe là bao nhiêu?"
]

def run(cmd, title=None):
    if title:
        print("\n" + "=" * 120)
        print(title)
        print("=" * 120)
    print("CMD:", " ".join(map(str, cmd)))
    r = subprocess.run(
        list(map(str, cmd)),
        text=True,
        encoding="utf-8",
        errors="replace",
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    print("RETURN CODE:", r.returncode)
    if r.stdout.strip():
        print("\n[STDOUT]")
        print(r.stdout)
    if r.stderr.strip():
        print("\n[STDERR]")
        print(r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(map(str, cmd))}")
    return r

# 1. Convert graph hybrid to retriever-compatible format.
# run([
#     sys.executable,
#     CONVERTER,
#     "--graph-root", GRAPH_ROOT,
#     "--output", COMPAT_GRAPH,
# ], "1. Convert graph hybrid v2 → LineARRAG compatible graph")

# 2. Build BM25 + graph index. No dense embedding, no GPU needed.
# run([
#     sys.executable,
#     RETRIEVER_DIR / "build_index.py",
#     "--graph-root", COMPAT_GRAPH,
#     "--gazetteer-root", GAZETTEER_ROOT,
#     "--output", INDEX_DIR,
#     "--skip-embeddings",
# ], "2. Build BM25 + graph index")

# 3. Test retrieval.
for q in QUERIES:
    run([
        sys.executable,
        RETRIEVER_DIR / "retrieve.py",
        "--index-dir", INDEX_DIR,
        "--gazetteer-root", GAZETTEER_ROOT,
        "--query", q,
        "--top-k", "20",

        "--dense-weight", "0.0",
        "--bm25-weight", "0.25",
        "--graph-weight", "0.15",
        "--reference-weight", "0.60",
    ], f"3. Retrieve: {q}")



3. Retrieve: Thời gian lái xe liên tục tối đa của người lái xe là bao nhiêu?
CMD: D:\Users\ADMIN\miniconda3\envs\kltn\python.exe D:\Users\ADMIN\LocalData\Classworks\KLTN\legal_linearrag_retriever\retrieve.py --index-dir D:\Users\ADMIN\LocalData\Classworks\KLTN\data\preprocessed\linearrag_index_hybrid_v2_bm25_graph_v2 --gazetteer-root D:\Users\ADMIN\LocalData\Classworks\KLTN\data\preprocessed\gazetteers_v2 --query Thời gian lái xe liên tục tối đa của người lái xe là bao nhiêu? --top-k 20 --dense-weight 0.0 --bm25-weight 0.25 --graph-weight 0.15 --reference-weight 0.60
RETURN CODE: 0

[STDOUT]
{
  "query": "Thời gian lái xe liên tục tối đa của người lái xe là bao nhiêu?",
  "activated_entities": [
    {
      "entity_id": "ent_62b35592536cfa2d",
      "score": 1.0,
      "canonical": "thời gian lái xe liên tục",
      "label": "CONDITION"
    },
    {
      "entity_id": "ent_d3f58a66c7c0be55",
      "score": 1.0,
      "canonical": "người lái xe",
      "label": "ACTOR"
    }
  ],
  "en